# Preprocesamiento de Datos — K-asar

**Dataset:** Sloan Digital Sky Survey - Data Release 17 (SDSS DR17).

**Autor:** Luis

**Fase:** Preprocesamiento de Datos

Este notebook aplica las decisiones metodológicas definidas en el Análisis Exploratorio de Datos (`01_exploratory_data_analysis.ipynb`), preparando el dataset para las fases posteriores de reducción de dimensionalidad (PCA) y clustering.

## 1. Configuración del entorno de trabajo

Se fija el directorio de trabajo en la raíz del repositorio para garantizar que las rutas relativas funcionen correctamente, y se importan las librerías necesarias para esta fase.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f"Directorio de trabajo actual: {os.getcwd()}")

Directorio de trabajo actual: c:\Users\SOMOS F5\Videos\Proyecto k-sar\k-sar


In [2]:
import pandas as pd
import numpy as np

## 2. Carga del dataset crudo

Se importa el archivo `stellar_classification.csv` tal como fue analizado en el EDA previo.

In [3]:
df_raw = pd.read_csv("data/stellar_classification.csv")
original_rows = df_raw.shape[0]

print(f"Dimensiones del dataset: {df_raw.shape}")
df_raw.head()

Dimensiones del dataset: (100000, 18)


,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842


## 3. Eliminación de columnas de metadatos

Según la conclusión 1 del EDA, se descartan 11 columnas identificadoras y de metadatos técnicos de captura (`obj_ID`, `alpha`, `delta`, `run_ID`, `rerun_ID`, `cam_col`, `field_ID`, `spec_obj_ID`, `plate`, `MJD`, `fiber_ID`), ya que carecen de poder discriminante astronómico para el clustering. La coordenadas celestes `alpha` y `delta` se incluyen en este descarte porque el EDA confirmó su nula capacidad para diferenciar tipos de objetos.

In [5]:
metadata_cols = ['obj_ID', 'alpha', 'delta', 'run_ID', 'rerun_ID',
                  'cam_col', 'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']

df = df_raw.drop(columns=[c for c in metadata_cols if c in df_raw.columns])

print(f"Columnas eliminadas: {len(metadata_cols)}")
print(f"Dimensiones tras eliminar metadatos: {df.shape}")
df.head()

Columnas eliminadas: 11
Dimensiones tras eliminar metadatos: (100000, 7)


,u,g,r,i,z,class,redshift
0,23.87882,22.27530,20.39501,19.16573,18.79371,GALAXY,0.634794
1,24.77759,22.83188,22.58444,21.16812,21.61427,GALAXY,0.779136
2,25.26307,22.66389,20.60976,19.34857,18.94827,GALAXY,0.644195
3,22.13682,23.77656,21.61162,20.50454,19.25010,GALAXY,0.932346
4,19.43718,17.58028,16.49747,15.97711,15.54461,GALAXY,0.116123


## 4. Aislamiento de la etiqueta real (`class`)

Según la conclusión 2 del EDA, la columna `class` se separa del resto de variables para mantener el clustering 100% no supervisado. Esta etiqueta se conserva únicamente para validar posteriormente si los clusters descubiertos coinciden con las categorías reales (estrella, galaxia, cuásar).

In [6]:
y_true = df['class'].copy()
X = df.drop(columns=['class'])

print(f"Distribución de clases reales (solo para validación posterior):")
print(y_true.value_counts())
print(f"\nDimensiones de features (X): {X.shape}")
X.head()

Distribución de clases reales (solo para validación posterior):
class
GALAXY    59445
STAR      21594
QSO       18961
Name: count, dtype: int64

Dimensiones de features (X): (100000, 6)


,u,g,r,i,z,redshift
0,23.87882,22.27530,20.39501,19.16573,18.79371,0.634794
1,24.77759,22.83188,22.58444,21.16812,21.61427,0.779136
2,25.26307,22.66389,20.60976,19.34857,18.94827,0.644195
3,22.13682,23.77656,21.61162,20.50454,19.25010,0.932346
4,19.43718,17.58028,16.49747,15.97711,15.54461,0.116123


## 5. Filtro de calidad: magnitudes fotométricas inválidas

Según la conclusión 3 del EDA, se eliminan los registros con magnitudes fotométricas ≤ 0 en las bandas u, g, r, i, z, ya que corresponden a errores de saturación del sensor CCD y no a mediciones astronómicas válidas.

In [7]:
magnitude_cols = ['u', 'g', 'r', 'i', 'z']

mask_invalid = (X[magnitude_cols] <= 0).any(axis=1)
n_invalid = int(mask_invalid.sum())

X = X[~mask_invalid].reset_index(drop=True)
y_true = y_true[~mask_invalid].reset_index(drop=True)

print(f"Registros eliminados por magnitud <= 0: {n_invalid}")
print(f"Dimensiones tras filtro de calidad: {X.shape}")

Registros eliminados por magnitud <= 0: 1
Dimensiones tras filtro de calidad: (99999, 6)


## 6. Ingeniería de características: índices de color

Según la conclusión 4 del EDA, se construyen 4 índices de color (u-g, g-r, r-i, i-z) que capturan la temperatura intrínseca de los cuerpos celestes y generan fronteras más claras entre agrupaciones físicas reales, según lo demostrado en el diagrama color-color del análisis exploratorio.

In [8]:
X_features = X.copy()

X_features['color_ug'] = X_features['u'] - X_features['g']
X_features['color_gr'] = X_features['g'] - X_features['r']
X_features['color_ri'] = X_features['r'] - X_features['i']
X_features['color_iz'] = X_features['i'] - X_features['z']

print(f"Dimensiones con índices de color añadidos: {X_features.shape}")
X_features.head()

Dimensiones con índices de color añadidos: (99999, 10)


,u,g,r,i,z,redshift,color_ug,color_gr,color_ri,color_iz
0,23.87882,22.27530,20.39501,19.16573,18.79371,0.634794,1.60352,1.88029,1.22928,0.37202
1,24.77759,22.83188,22.58444,21.16812,21.61427,0.779136,1.94571,0.24744,1.41632,-0.44615
2,25.26307,22.66389,20.60976,19.34857,18.94827,0.644195,2.59918,2.05413,1.26119,0.40030
3,22.13682,23.77656,21.61162,20.50454,19.25010,0.932346,-1.63974,2.16494,1.10708,1.25444
4,19.43718,17.58028,16.49747,15.97711,15.54461,0.116123,1.85690,1.08281,0.52036,0.43250


## 7. Escalado robusto de las variables

Según la conclusión 5 del EDA, se aplica `RobustScaler` en lugar de `StandardScaler` o `MinMaxScaler`. Esta decisión se justifica porque MinMaxScaler comprimiría el 99% de los datos en un rango mínimo debido a los outliers de alto redshift (cuásares con z > 3), y StandardScaler se distorsiona por la misma asimetría al depender de la media y la desviación estándar. RobustScaler utiliza la mediana y el rango intercuartílico (IQR), garantizando que la estructura central de estrellas y galaxias permanezca correctamente escalada e insensible a las colas extremas.

In [9]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled_array = scaler.fit_transform(X_features)

X_scaled = pd.DataFrame(X_scaled_array, columns=X_features.columns, index=X_features.index)

print(f"Dimensiones del dataset escalado: {X_scaled.shape}")
X_scaled.describe().T[['min', '25%', '50%', '75%', 'max']]

Dimensiones del dataset escalado: (99999, 10)


,min,25%,50%,75%,max
u,-3.353126,-0.547734,0.0,0.452266,3.179019
g,-3.356534,-0.675848,0.0,0.324152,3.325057
r,-3.541856,-0.683918,0.0,0.316082,3.247359
i,-3.729125,-0.627900,0.0,0.372100,4.780488
z,-3.817545,-0.627475,0.0,0.372525,4.218665
redshift,-0.668277,-0.569004,0.0,0.430996,10.139407
color_ug,-9.955176,-0.488541,0.0,0.511459,12.242974
color_gr,-11.058543,-0.460409,0.0,0.539591,11.170079
color_ri,-22.383793,-0.389830,0.0,0.610170,17.349105
color_iz,-39.756523,-0.599358,0.0,0.400642,38.544879


## 8. Reporte de trazabilidad del preprocesamiento

Se documenta cuantitativamente cada transformación aplicada: filas originales, columnas de metadatos eliminadas, registros descartados por calidad, e índices de color generados, garantizando la trazabilidad completa del pipeline.

In [10]:
final_rows = X_scaled.shape[0]

report = pd.DataFrame({
    'metric': [
        'filas_originales',
        'columnas_metadatos_eliminadas',
        'registros_eliminados_magnitud_invalida',
        'indices_color_generados',
        'filas_finales',
        'columnas_finales'
    ],
    'value': [
        original_rows,
        len(metadata_cols),
        n_invalid,
        4,
        final_rows,
        X_scaled.shape[1]
    ]
})

report

,metric,value
0,filas_originales,100000
1,columnas_metadatos_eliminadas,11
2,registros_eliminados_magnitud_invalida,1
3,indices_color_generados,4
4,filas_finales,99999
5,columnas_finales,10


## 9. Exportación de entregables finales

Se generan tres archivos en `data/`: el dataset escalado listo para PCA (`stellar_scaled.csv`), la etiqueta real reservada para validación posterior (`stellar_labels.csv`), y el reporte de trazabilidad (`preprocessing_report.csv`).

In [11]:
X_scaled.to_csv("data/stellar_scaled.csv", index=False)
y_true.to_csv("data/stellar_labels.csv", index=False)
report.to_csv("data/preprocessing_report.csv", index=False)

print("Archivos exportados correctamente en data/:")
print("- stellar_scaled.csv")
print("- stellar_labels.csv")
print("- preprocessing_report.csv")

Archivos exportados correctamente en data/:
- stellar_scaled.csv
- stellar_labels.csv
- preprocessing_report.csv
